# NER MacBERT 上手演示（单文档、零文件写入）

这个 Notebook 面向第一次接触 `ner_macbert_trainer` 的同学：
- 跑通一遍完整链路（数据检查 -> 训练侧样本构造 -> 模型前向 -> ONNX推理 -> 双模型融合）
- 只处理 1 个文档，帮助建立 IO 的具象理解
- 全过程只读不写，不生成任何新文件，避免污染工程结构

建议结合阅读：
- `/home/superuser/dev/NER/ner_macbert_trainer/sop/Model_Optimize.md`
- `/home/superuser/dev/NER/ner_macbert_trainer/sop/MANUAL_MACBERT_NER.md`

## Step 1: 环境与路径初始化
这一步只做导入与路径检查。

In [1]:
from pathlib import Path
import sys
import json
import yaml
import numpy as np
import onnxruntime as ort

PROJECT_ROOT = Path('/home/superuser/dev/NER/ner_macbert_trainer')
UPSTREAM_ROOT = Path('/home/superuser/dev/NER/ner_dataset_builder')
DATA_DIR = Path('/home/superuser/dev/NER/data')
CONFIG_PATH = PROJECT_ROOT / 'conf' / 'training_args.yaml'
SAVED_MODEL_DIR = PROJECT_ROOT / 'output' / 'saved_models'
ONNX_INT8_PATH = PROJECT_ROOT / 'output' / 'onnx_models' / 'ner_macbert_int8.onnx'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('UPSTREAM_ROOT =', UPSTREAM_ROOT)
print('DATA_DIR exists =', DATA_DIR.exists())
print('CONFIG_PATH exists =', CONFIG_PATH.exists())
print('SAVED_MODEL_DIR exists =', SAVED_MODEL_DIR.exists())
print('ONNX_INT8_PATH exists =', ONNX_INT8_PATH.exists())

PROJECT_ROOT = /home/superuser/dev/NER/ner_macbert_trainer
UPSTREAM_ROOT = /home/superuser/dev/NER/ner_dataset_builder
DATA_DIR exists = True
CONFIG_PATH exists = True
SAVED_MODEL_DIR exists = True
ONNX_INT8_PATH exists = True


## Step 2: 读取训练配置，明确关键 IO
确认当前训练主路径、底模路径、运行参数。

In [2]:
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print(json.dumps({
    'train_bio_path': cfg['data']['train_bio_path'],
    'pretrained_model_path': cfg['model']['pretrained_model_path'],
    'saved_model_dir': cfg['output']['saved_model_dir'],
    'onnx_dir': cfg['output']['onnx_dir'],
    'max_length': cfg['runtime']['max_length'],
    'train_window_stride': cfg['runtime'].get('train_window_stride'),
    'infer_window_stride': cfg['runtime'].get('infer_window_stride'),
    'split_by_source_file': cfg['runtime'].get('split_by_source_file'),
    'strip_table_blocks': cfg['runtime'].get('strip_table_blocks')
}, ensure_ascii=False, indent=2))

{
  "train_bio_path": "/home/superuser/dev/NER/ner_dataset_builder/output/train.bio",
  "pretrained_model_path": "/home/superuser/LLM_Model/chinese-macbert-large",
  "saved_model_dir": "/home/superuser/dev/NER/ner_macbert_trainer/output/saved_models",
  "onnx_dir": "/home/superuser/dev/NER/ner_macbert_trainer/output/onnx_models",
  "max_length": 256,
  "train_window_stride": 64,
  "infer_window_stride": 64,
  "split_by_source_file": true,
  "strip_table_blocks": true
}


## Step 3: 从输入目录读取 1 个文档（只读）
这里使用推理侧同款加载逻辑，确保你看到的是线上同口径输入。

In [3]:
from inference_onnx import load_items_from_directory

all_items = load_items_from_directory(str(DATA_DIR))
assert all_items, '未加载到任何 page_text'

one_source_file = all_items[0]['source_file']
doc_items = [x for x in all_items if x['source_file'] == one_source_file]
sample_item = doc_items[0]

print('source_file =', one_source_file)
print('doc_page_text_count =', len(doc_items))
print('sample page_number =', sample_item['page_number'])
print('sample block_index =', sample_item['block_index'])
print('sample text_length =', len(sample_item['text']))
print('\ntext_preview:\n', sample_item['text'][:300])

source_file = SN0006-01 井录井报告.md
doc_page_text_count = 37
sample page_number = 1
sample block_index = 0
sample text_length = 51

text_preview:
 # <br> 鄂尔多斯盆地伊陕斜坡 <br>

# <br> SN0006-01 井录井报告 <br>


## Step 4: 数据健康检查（训练语料层）
这一步对应 SOP 中“先看数据再谈调参”的原则。

In [4]:
from core.dataset import parse_bio_file
from collections import Counter

train_bio_path = cfg['data']['train_bio_path']
sentences, tags = parse_bio_file(train_bio_path)

all_tokens = sum(len(x) for x in tags)
entity_tokens = sum(sum(1 for t in row if t != 'O') for row in tags)
label_counter = Counter(t for row in tags for t in row)

print('sentence_count =', len(sentences))
print('entity_ratio =', round(entity_tokens / all_tokens, 6))
print('top_labels =', label_counter.most_common(10))

sentence_count = 5689
entity_ratio = 0.002979
top_labels = [('O', 294502), ('I-block', 268), ('I-well_name', 250), ('I-strat_unit', 209), ('B-strat_unit', 74), ('B-block', 39), ('B-well_name', 32), ('B-lithology', 4), ('I-lithology', 4)]


## Step 5: 训练侧样本如何进入模型（窗口化）
不做全量训练，只演示“句子 -> token对齐 -> window样本”的关键 IO。

In [5]:
from transformers import AutoTokenizer
from core.dataset import build_label_mappings, NERDataset

label2id, id2label = build_label_mappings(tags)
tokenizer = AutoTokenizer.from_pretrained(cfg['model']['pretrained_model_path'], use_fast=True)

mini_sentences = sentences[:8]
mini_tags = tags[:8]

mini_ds = NERDataset(
    mini_sentences,
    mini_tags,
    tokenizer,
    label2id,
    max_length=int(cfg['runtime']['max_length']),
    window_stride=int(cfg['runtime'].get('train_window_stride', 64)),
)

sample = mini_ds[0]
print('mini_windows =', len(mini_ds))
print('keys =', list(sample.keys()))
print('input_ids_len =', len(sample['input_ids']))
print('labels_len =', len(sample['labels']))
print('first_20_labels =', sample['labels'][:20])

mini_windows = 8
keys = ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
input_ids_len = 21
labels_len = 21
first_20_labels = [-100, 8, 8, 8, 8, 8, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8]


## Step 6: 加载已训练模型，跑一次前向
这一步帮助你把“数据张量 -> logits”建立直觉。

In [6]:
import torch
from transformers import AutoModelForTokenClassification

assert SAVED_MODEL_DIR.exists(), '请先确保已有训练完成的 saved_models'
model = AutoModelForTokenClassification.from_pretrained(str(SAVED_MODEL_DIR))
model.eval()

batch = {k: torch.tensor([v], dtype=torch.long) for k, v in sample.items() if k in {'input_ids', 'attention_mask', 'token_type_ids'}}
with torch.no_grad():
    out = model(**batch)

print('logits_shape =', tuple(out.logits.shape))
print('num_labels =', out.logits.shape[-1])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

logits_shape = (1, 21, 9)
num_labels = 9


## Step 7: 单文档 ONNX 推理（内存中）
只跑当前文档的 1 条 page_text，不落盘，直接观察预测标签和 span。

In [7]:
from inference_onnx import load_id2label, predict_tags_for_text, normalize_bio_tags, tags_to_spans, apply_span_constraints, apply_texture_lexicon

assert ONNX_INT8_PATH.exists(), '请先确保 ONNX 已导出'

session = ort.InferenceSession(str(ONNX_INT8_PATH), providers=['CPUExecutionProvider'])
id2label = load_id2label(SAVED_MODEL_DIR)

text = sample_item['text']
tags_pred = predict_tags_for_text(
    text=text,
    tokenizer=tokenizer,
    session=session,
    id2label=id2label,
    max_length=int(cfg['runtime']['max_length']),
    window_stride=int(cfg['runtime'].get('infer_window_stride', 64)),
)
tags_pred = normalize_bio_tags(tags_pred)
tags_pred = apply_span_constraints(tags_pred, text, cfg['runtime'].get('span_constraints', {}))
tags_pred = apply_texture_lexicon(tags_pred, text, cfg['runtime'].get('texture_lexicon', []))
spans = tags_to_spans(tags_pred, text)

print('text_len =', len(text), 'tag_len =', len(tags_pred))
print('non_O_count =', sum(1 for t in tags_pred if t != 'O'))
print('top_spans =', spans[:10])

text_len = 51 tag_len = 51
non_O_count = 9
top_spans = [('strat_unit', 8, 8, '尔'), ('strat_unit', 10, 16, '斯盆地伊陕斜坡'), ('lithology', 31, 31, 'S')]


## Step 8: 双模型融合演示（可选）
如果双模型 ONNX 存在，则演示融合规则如何影响纹理标签。

In [8]:
from inference_onnx_dual import fuse_tags
from inference_onnx import load_config

base_cfg_path = PROJECT_ROOT / 'conf' / 'training_args.dual_base.yaml'
texture_cfg_path = PROJECT_ROOT / 'conf' / 'training_args.dual_texture.yaml'

if base_cfg_path.exists() and texture_cfg_path.exists():
    base_cfg = load_config(str(base_cfg_path))
    tex_cfg = load_config(str(texture_cfg_path))
    base_onnx = Path(base_cfg['output']['onnx_dir']) / 'ner_macbert_int8.onnx'
    tex_onnx = Path(tex_cfg['output']['onnx_dir']) / 'ner_macbert_int8.onnx'
    base_saved = Path(base_cfg['output']['saved_model_dir'])
    tex_saved = Path(tex_cfg['output']['saved_model_dir'])

    if base_onnx.exists() and tex_onnx.exists() and base_saved.exists() and tex_saved.exists():
        base_sess = ort.InferenceSession(str(base_onnx), providers=['CPUExecutionProvider'])
        tex_sess = ort.InferenceSession(str(tex_onnx), providers=['CPUExecutionProvider'])
        base_id2 = load_id2label(base_saved)
        tex_id2 = load_id2label(tex_saved)
        base_tags = normalize_bio_tags(predict_tags_for_text(text, tokenizer, base_sess, base_id2, int(cfg['runtime']['max_length']), int(cfg['runtime'].get('infer_window_stride', 64))))
        tex_tags = normalize_bio_tags(predict_tags_for_text(text, tokenizer, tex_sess, tex_id2, int(cfg['runtime']['max_length']), int(cfg['runtime'].get('infer_window_stride', 64))))
        fused_tags = fuse_tags(base_tags, tex_tags)
        print('base_non_O =', sum(1 for t in base_tags if t != 'O'))
        print('texture_non_O =', sum(1 for t in tex_tags if t != 'O'))
        print('fused_non_O =', sum(1 for t in fused_tags if t != 'O'))
        print('fused_spans_top =', tags_to_spans(fused_tags, text)[:10])
    else:
        print('跳过：双模型ONNX或saved_models尚未准备齐全')
else:
    print('跳过：未找到双模型配置文件')

base_non_O = 16
texture_non_O = 19
fused_non_O = 16
fused_spans_top = [('block', 7, 8, '鄂尔'), ('block', 10, 16, '斯盆地伊陕斜坡'), ('well_name', 32, 38, 'N0006-0')]


## Step 9: 新手应形成的 IO 心智图
最后一步把本工程最关键 IO 关系做成可读摘要，便于结合 SOP 深入。

In [9]:
summary = {
    'input_train_bio': cfg['data']['train_bio_path'],
    'train_entry': str(PROJECT_ROOT / 'main_train.py'),
    'model_artifact_dir': cfg['output']['saved_model_dir'],
    'onnx_export_entry': str(PROJECT_ROOT / 'export_onnx.py'),
    'onnx_int8_path': str(ONNX_INT8_PATH),
    'infer_entry': str(PROJECT_ROOT / 'inference_onnx.py'),
    'dual_infer_entry': str(PROJECT_ROOT / 'inference_onnx_dual.py'),
    'sop_optimize': str(PROJECT_ROOT / 'sop' / 'Model_Optimize.md'),
    'sop_manual': str(PROJECT_ROOT / 'sop' / 'MANUAL_MACBERT_NER.md')
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

print('\n建议下一步：')
print('1) 对照 Model_Optimize.md 的原因分解，先做数据健康检查')
print('2) 每次只改一个变量，记录 test_metrics 快照')
print('3) 与上游对齐数据版本说明，再比较模型结果')

{
  "input_train_bio": "/home/superuser/dev/NER/ner_dataset_builder/output/train.bio",
  "train_entry": "/home/superuser/dev/NER/ner_macbert_trainer/main_train.py",
  "model_artifact_dir": "/home/superuser/dev/NER/ner_macbert_trainer/output/saved_models",
  "onnx_export_entry": "/home/superuser/dev/NER/ner_macbert_trainer/export_onnx.py",
  "onnx_int8_path": "/home/superuser/dev/NER/ner_macbert_trainer/output/onnx_models/ner_macbert_int8.onnx",
  "infer_entry": "/home/superuser/dev/NER/ner_macbert_trainer/inference_onnx.py",
  "dual_infer_entry": "/home/superuser/dev/NER/ner_macbert_trainer/inference_onnx_dual.py",
  "sop_optimize": "/home/superuser/dev/NER/ner_macbert_trainer/sop/Model_Optimize.md",
  "sop_manual": "/home/superuser/dev/NER/ner_macbert_trainer/sop/MANUAL_MACBERT_NER.md"
}

建议下一步：
1) 对照 Model_Optimize.md 的原因分解，先做数据健康检查
2) 每次只改一个变量，记录 test_metrics 快照
3) 与上游对齐数据版本说明，再比较模型结果
